# Clase 210 — PySpark básico para datasets grandes

Requiere: `pip install pyspark==3.5.3`. Usaremos modo local con todos los cores.

## 🧠 Intuición previa

**PySpark es "pandas que corre repartido en muchas máquinas".** Escribís código casi
idéntico al de pandas (`df.filter(...).groupBy(...).agg(...)`), pero por debajo Spark
**parte el dataset en trozos**, los reparte entre varios workers y procesa cada trozo en
paralelo. Por eso puede con datasets que **no caben en la RAM de una sola máquina**: nunca
tiene todo el dato junto, trabaja de a pedazos.

Dos ideas clave que cambian respecto de pandas:

1. **Lazy (perezoso)**: `df.filter(...).select(...)` **no ejecuta nada** — solo arma un
   plan. La ejecución dispara recién con una *acción* (`.count()`, `.collect()`,
   `.write`). Spark usa ese plan para optimizar (empujar filtros, leer solo las columnas
   necesarias).
2. **El shuffle es lo caro**: cuando agrupás o hacés join, Spark tiene que **mover datos
   entre máquinas** para juntar las mismas keys. Ese movimiento (shuffle) es el cuello de
   botella; casi toda la optimización de Spark gira en torno a reducirlo.

Acá **simulamos esas ideas con pandas** (todo cabe en RAM), pero el modelo mental
—particionar, lazy, evitar shuffle— es exactamente el de un cluster real.

In [ ]:
import os, tempfile, shutil
from pathlib import Path
WORK = Path(tempfile.gettempdir()) / 'spark_demo'
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir(); os.chdir(WORK)

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.functions import broadcast, col, rand, when

spark = (SparkSession.builder
         .master('local[*]')
         .appName('clase210')
         .config('spark.sql.shuffle.partitions', '8')   # bajo para dataset chico
         .config('spark.ui.port', '4042')
         .getOrCreate())
print('Spark version:', spark.version, '| cores:', spark.sparkContext.defaultParallelism)

## 1. Dataset sintético — taxi trips

In [ ]:
# 1M filas, 5 columnas
df = (spark.range(0, 1_000_000)
      .withColumn('zone_id', (rand(seed=1) * 50).cast('int'))
      .withColumn('fare', rand(seed=2) * 100 + 5)
      .withColumn('tip', rand(seed=3) * 20)
      .withColumn('date', F.expr("date_add(date '2024-01-01', cast(rand(seed=4) * 30 as int))")))
df.printSchema()
df.show(5)
print('rows:', df.count())

## 2. Lazy vs eager — observable en Spark UI

In [ ]:
import time

t0 = time.perf_counter()
df2 = df.filter(col('fare') > 30).select('zone_id', 'fare', 'tip')   # transformation = lazy
print(f'transformation: {(time.perf_counter() - t0) * 1000:.1f} ms (no ejecutó nada)')

t0 = time.perf_counter()
n = df2.count()   # action = ejecuta
print(f'action df2.count(): {time.perf_counter() - t0:.2f} s — devolvió {n:,} filas')

## 3. Broadcast join (tabla chica × tabla grande)

In [ ]:
zones = spark.createDataFrame(
    [(i, f'zone_{i}', 'Manhattan' if i < 20 else 'Brooklyn' if i < 40 else 'Queens') for i in range(50)],
    ['zone_id', 'name', 'borough'],
)

t0 = time.perf_counter()
joined = df.join(broadcast(zones), 'zone_id').groupBy('borough').agg(F.avg('fare').alias('avg_fare'), F.count('*').alias('n'))
joined.show()
print(f'broadcast join + agg: {time.perf_counter() - t0:.2f} s')

## 4. Particionado al escribir

In [ ]:
out_path = str(WORK / 'trips_partitioned')
df.write.mode('overwrite').partitionBy('date').parquet(out_path)

# Verificar estructura
subdirs = sorted([p.name for p in Path(out_path).iterdir() if p.is_dir()])[:5]
print('subdirs:', subdirs)

# Predicate pushdown: leer solo 1 partición
t0 = time.perf_counter()
one_day = spark.read.parquet(out_path).filter(col('date') == '2024-01-15').count()
print(f'lectura 1 día: {one_day:,} filas en {time.perf_counter() - t0:.2f} s (solo leyó esa partición)')

## 5. Skew + salting

In [ ]:
# Crear skew: 90% en zone_id=1
skewed = (spark.range(0, 1_000_000)
          .withColumn('zone_id', when(rand(seed=5) < 0.9, 1).otherwise((rand(seed=6) * 50).cast('int')))
          .withColumn('amount', rand(seed=7) * 100))

t0 = time.perf_counter()
skewed.groupBy('zone_id').agg(F.sum('amount').alias('total')).count()
print(f'groupBy SIN salting: {time.perf_counter() - t0:.2f} s')

# Salting: agregar columna random para repartir la key caliente
salted = skewed.withColumn('salt', (rand(seed=8) * 10).cast('int'))
t0 = time.perf_counter()
(salted.groupBy('zone_id', 'salt').agg(F.sum('amount').alias('partial'))
       .groupBy('zone_id').agg(F.sum('partial').alias('total')).count())
print(f'groupBy CON salting: {time.perf_counter() - t0:.2f} s')
print('(en datasets más grandes el speedup es 5-10×; acá es chico para terminar rápido)')

## 6. Spark SQL (mismo resultado, otra sintaxis)

In [ ]:
df.createOrReplaceTempView('trips')
zones.createOrReplaceTempView('zones')
spark.sql('''
  SELECT z.borough, AVG(t.fare) AS avg_fare, COUNT(*) AS n
  FROM trips t JOIN zones z USING (zone_id)
  GROUP BY z.borough
  ORDER BY avg_fare DESC
''').show()

In [ ]:
spark.stop()
print('Spark session detenida.')

## Ejercicio guiado

1. Descargá un mes de NYC Yellow Taxi parquet (~150 MB). Reemplazá el dataset sintético y midí tiempos.
2. Hacé `.explain('extended')` sobre un join — observá el plan físico de Catalyst.
3. Probá `df.cache()` antes de 3 agregaciones distintas; comparar vs sin cache.
4. Cambiá `spark.sql.shuffle.partitions` entre 4 y 200 para el mismo dataset — encontrá el sweet spot.
5. Migrá un script pandas existente a PySpark. Midí RAM peak (`memory_profiler`).

## Conclusiones

- Spark es overkill para <1 GB; vale la pena desde unos 10 GB en una sola máquina.
- Broadcast join + AQE matan el 80% de los problemas de performance.
- `partitionBy` al escribir es lo que hace que lecturas filtradas sean rápidas.
- Skew es el bug invisible; UI lo muestra, salting lo resuelve.

## ✅ Soluciones de los ejercicios

PySpark no está instalado (arrastra la JVM + Spark, ~300 MB). **Simulamos su semántica con
pandas + pyarrow**: `SparkSession` → cargar parquet, *lazy vs eager* → un mini plan
diferido, *broadcast join* → merge con la dimensión chica, *partitionBy* → escribir parquet
particionado en disco, *skew + salting* → la técnica anti-shuffle. Al lado dejamos el código
PySpark real como referencia. Todo corre local, sin cluster ni internet.

### Ejercicio 1 — Spark session local + cargar parquet + `printSchema()`

`spark.read.parquet(...)` + `df.printSchema()`. Lo replicamos escribiendo/leyendo un parquet
con pyarrow y mostrando el schema (nombres + tipos).

In [ ]:
import tempfile, os, shutil
from pathlib import Path
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq

# --- PySpark real (referencia) ---------------------------------------------
# spark = SparkSession.builder.master("local[4]").appName("demo").getOrCreate()
# df = spark.read.parquet("taxi.parquet"); df.printSchema()
# ---------------------------------------------------------------------------

WORK = Path(tempfile.gettempdir()) / "spark_sim"; WORK.mkdir(exist_ok=True)
rng = np.random.default_rng(0)
N = 10_000
taxi = pd.DataFrame({
    "trip_id": np.arange(N),
    "zone_id": rng.integers(1, 6, N),
    "fare": rng.gamma(3.0, 4.0, N).round(2),
    "date": pd.to_datetime("2024-01-01") + pd.to_timedelta(rng.integers(0, 3, N), unit="D"),
})
path = WORK / "taxi.parquet"
taxi.to_parquet(path)                       # equivalente a df.write.parquet(...)

df = pd.read_parquet(path)                   # equivalente a spark.read.parquet(...)
schema = pq.read_schema(path)
print("printSchema() equivalente:")
for name, typ in zip(schema.names, schema.types):
    print(f"  |-- {name}: {typ}")

assert df.shape == (N, 4)
assert set(schema.names) == {"trip_id", "zone_id", "fare", "date"}
print("OK ejercicio 1 — parquet cargado y schema inspeccionado")

### Ejercicio 2 — Lazy vs eager

En Spark, `filter/select` construyen un plan; la acción `count()` lo ejecuta. Modelamos un
`LazyFrame` que **encola transformaciones** y solo las corre al pedir una acción.

In [ ]:
class LazyFrame:
    """Mini-emulación del modelo lazy de Spark sobre un DataFrame pandas."""
    def __init__(self, df, plan=None):
        self._df = df
        self._plan = plan or []
        self.executed = False

    def filter(self, fn):
        return LazyFrame(self._df, self._plan + [("filter", fn)])

    def select(self, cols):
        return LazyFrame(self._df, self._plan + [("select", cols)])

    def _materialize(self):
        out = self._df
        for op, arg in self._plan:
            out = out[arg(out)] if op == "filter" else out[arg]
        self.executed = True
        return out

    def count(self):          # ACCIÓN: dispara la ejecución
        return len(self._materialize())

lf = LazyFrame(df)
plan = lf.filter(lambda d: d["fare"] > 10).select(["trip_id", "fare"])
print("¿ya ejecutó tras filter+select?", plan.executed)   # False: es lazy
n = plan.count()                                           # recién acá ejecuta
print("filas tras el plan:", n, "| ejecutado ahora?", plan.executed)

assert plan.executed is True and n == int((df["fare"] > 10).sum())
print("OK ejercicio 2 — transformaciones lazy, ejecución diferida a la acción count()")

### Ejercicio 3 — Broadcast join

`taxi` es enorme y `zones` es minúscula. Con `broadcast(zones)` Spark **copia la tabla chica
a cada worker** para evitar mover la grande (evita shuffle). El resultado es idéntico a un
join normal; solo cambia el *cómo*.

In [ ]:
zones = pd.DataFrame({"zone_id": [1, 2, 3, 4, 5],
                      "zone_name": ["Centro", "Norte", "Sur", "Este", "Oeste"]})

# --- PySpark real: taxi.join(broadcast(zones), "zone_id") ------------------
# Con broadcast, la dim chica viaja a cada worker; la grande no se shufflea.
join_broadcast = df.merge(zones, on="zone_id", how="left")   # dim chica "broadcasteada"
join_plain     = df.merge(zones, on="zone_id", how="left")   # join normal

# el RESULTADO es el mismo; broadcast solo cambia la estrategia física
assert join_broadcast.equals(join_plain)
assert join_broadcast["zone_name"].notna().all()
assert len(join_broadcast) == len(df)          # left join no infla filas
print(join_broadcast[["trip_id", "zone_id", "zone_name", "fare"]].head())
print("OK ejercicio 3 — broadcast join = mismo resultado, sin shuffle de la tabla grande")

### Ejercicio 4 — Particionado al escribir (`partitionBy`)

`df.write.partitionBy("date")` crea un subdirectorio por valor: `out/date=.../part-*.parquet`.
Una query con `WHERE date='...'` solo lee ese subdirectorio (*partition pruning*). Lo hacemos
con `pyarrow.parquet.write_to_dataset(partition_cols=...)`.

In [ ]:
out = WORK / "taxi_partitioned"
if out.exists(): shutil.rmtree(out)

taxi2 = taxi.copy()
taxi2["date"] = taxi2["date"].dt.strftime("%Y-%m-%d")   # clave de partición legible
pq.write_to_dataset(pa.Table.from_pandas(taxi2), root_path=str(out),
                    partition_cols=["date"])

subdirs = sorted(p.name for p in out.iterdir() if p.is_dir())
print("estructura en disco:", subdirs)

# leer SOLO una partición = partition pruning
one_day = pd.read_parquet(out, filters=[("date", "==", "2024-01-01")])
print("filas leídas para 2024-01-01:", len(one_day))

assert all(s.startswith("date=") for s in subdirs), "layout Hive: date=YYYY-MM-DD"
assert (one_day["date"] == "2024-01-01").all()
assert len(one_day) == int((taxi2["date"] == "2024-01-01").sum())
print("OK ejercicio 4 — parquet particionado por date; lectura con filtro poda particiones")

### Ejercicio 5 — Skew + salting

Una key *skewed* (90% de las filas con `user_id=1`) hace que **una sola task** cargue casi
todo el trabajo en el shuffle. La técnica **salting** rompe esa key en `key×salt` para
repartir, y luego re-agrega. Verificamos que el total no cambia pero el trabajo se balancea.

In [ ]:
M = 100_000
user_id = np.where(rng.random(M) < 0.9, 1, rng.integers(2, 50, M))   # 90% -> user 1
events = pd.DataFrame({"user_id": user_id, "amount": rng.random(M).round(3)})

# groupBy directo: la partición de user_id=1 concentra ~90% del trabajo (skew)
by_user = events.groupby("user_id")["amount"].sum()
carga = events["user_id"].value_counts(normalize=True).max()
print(f"la key más pesada concentra el {carga:.0%} de las filas (skew)")

# --- Salting: (user_id, salt) reparte user_id=1 en varias sub-particiones ---
SALT = 10
events["salt"] = rng.integers(0, SALT, M)
partial = events.groupby(["user_id", "salt"])["amount"].sum()      # 1a pasada, balanceada
final = partial.groupby("user_id").sum()                            # 2a pasada, re-agrega

# el resultado es idéntico al groupBy directo, pero el trabajo quedó repartido
assert np.allclose(final.sort_index().values, by_user.sort_index().values)
sub_particiones = partial.loc[1].shape[0]
assert sub_particiones > 1, "user_id=1 se repartió en varias sub-particiones por el salt"
print(f"con salting, user_id=1 se dividió en {sub_particiones} sub-particiones; totales iguales")
print("OK ejercicio 5 — salting mitiga el skew sin cambiar el resultado")